In [1]:
#sql analysis
import sqlite3
import pandas as pd

# Create SQLite database connection
conn = sqlite3.connect('work_health_project.db')


# Reload all cleaned datasets from saved CSV files
final_merged_df = pd.read_csv('Cleaned_datasets/final_merged_burnout_oecd.csv')
sleep_df = pd.read_csv('Cleaned_datasets/sleep_df.csv')
oecd_wide = pd.read_csv('Cleaned_datasets/oecd_wide_country_data.csv')

print("All datasets loaded!")
print("Burnout shape:", final_merged_df.shape)
print("Sleep shape:", sleep_df.shape)
print("OECD shape:", oecd_wide.shape)

# Load all cleaned datasets into SQL tables
final_merged_df.to_sql('burnout', conn, if_exists='replace', index=False)
sleep_df.to_sql('sleep', conn, if_exists='replace', index=False)
oecd_wide.to_sql('oecd', conn, if_exists='replace', index=False)

print("Database created!")
print("Tables loaded: burnout, sleep, oecd")

# Verify tables exist
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tables in database:", cursor.fetchall())

All datasets loaded!
Burnout shape: (3000, 37)
Sleep shape: (374, 18)
OECD shape: (41, 10)
Database created!
Tables loaded: burnout, sleep, oecd
Tables in database: [('burnout',), ('sleep',), ('oecd',)]


In [2]:
def run_query(query, description=""):
    print(f"\n{'='*50}")
    print(f"📊 {description}")
    print(f"{'='*50}")
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))
    return result

In [3]:
q1=run_query("""select job_role, round(avg(burnout_level),2) as avg_burnout,
round(avg(stress_level),2) as avg_stress, count(*) as total_employees
from burnout 
group by job_role
order by avg_burnout desc 
limit 5""", "top 5 most burned out job roles")


📊 top 5 most burned out job roles
         job_role  avg_burnout  avg_stress  total_employees
    HR Specialist         5.64        5.49              361
Software Engineer         5.61        5.30              401
  Sales Associate         5.55        5.66              355
  Project Manager         5.51        5.71              362
   Data Scientist         5.48        5.48              411


In [4]:
q2=run_query(""" select department,round(avg(stress_level),2) as avg_stress,
round(avg(burnout_level),2) as avg_burnout_level,count(*) as total_employee
from burnout
group by department
order by 2 desc
limit 5""","top 5 department wise avg stress and burnout")


📊 top 5 department wise avg stress and burnout
 department  avg_stress  avg_burnout_level  total_employee
         IT        5.70               5.52             482
Engineering        5.65               5.45             488
  Marketing        5.60               5.50             521
      Sales        5.40               5.57             522
         HR        5.39               5.61             525


In [5]:
q3=run_query(""" select country, round(avg(work_hours_per_week),2) as avg_work_hours,
round(avg(work_life_balance_score),2) as avg_work_life_balance,
round(avg(job_satisfaction),2) as avg_job_satisfaction
from burnout 
group by country
order by avg_work_hours desc
limit 10""","top 10 countries having highest work hours per week")


📊 top 10 countries having highest work hours per week
       country  avg_work_hours  avg_work_life_balance  avg_job_satisfaction
United Kingdom           44.81                   5.53                  5.30
        Brazil           44.74                   5.22                  5.32
     Australia           44.56                   5.54                  5.40
         India           44.46                   5.45                  5.46
        Canada           44.36                   5.61                  5.61
 United States           44.29                   5.35                  5.50
       Germany           44.29                   5.45                  5.48


In [6]:
q4=run_query(""" select remote_work, round(avg(productivity_score),2) as avg_productivity
from burnout
group by remote_work
order by avg_productivity desc""","remote work impact on productivity")



📊 remote work impact on productivity
remote_work  avg_productivity
     Hybrid              5.63
         No              5.49
        Yes              5.43


In [7]:
q5=run_query(""" select department, round(avg(stress_level),2) as avg_stress,
                 ROUND(AVG(burnout_level), 2) AS avg_burnout,
                 ROUND(AVG(job_satisfaction), 2) AS avg_job_satisfaction,
                ROUND(AVG(sleep_hours), 2) AS avg_sleep_hours,
                 ROUND(AVG(physical_activity_hrs), 2) AS avg_physical_activity,
                 salary_range,
                 COUNT(*) AS total_employees,
                 SUM(CASE WHEN burnout_risk = 1 THEN 1 ELSE 0 END) AS high_risk_count
              FROM burnout
             GROUP BY department, salary_range
             ORDER BY avg_stress DESC
             """, "Department Health & Burnout Summary")


📊 Department Health & Burnout Summary
 department  avg_stress  avg_burnout  avg_job_satisfaction  avg_sleep_hours  avg_physical_activity salary_range  total_employees  high_risk_count
         IT        6.05         5.92                  5.56             6.31                   5.01      60K-80K               97               41
      Sales        5.93         5.60                  5.75             6.64                   5.47      40K-60K              104               31
Engineering        5.91         5.24                  5.65             6.39                   5.09      60K-80K              102               28
    Support        5.82         5.41                  5.51             6.86                   5.21      40K-60K               91               27
  Marketing        5.81         5.66                  5.83             6.64                   5.00         <40K              104               37
         IT        5.80         5.55                  5.62             6.72          

In [8]:
q6= run_query("""
    SELECT
        occupation,
        ROUND(AVG(age), 1) AS avg_age,
        ROUND(AVG(stress_level), 2) AS avg_stress,
        ROUND(AVG(physical_activity_level), 2) AS avg_physical_activity,
        ROUND(AVG(quality_of_sleep), 2) AS avg_sleep_quality,
        bmi_category,
        COUNT(*) AS total_people,
        SUM(CASE WHEN sleep_disorder != 'None' THEN 1 ELSE 0 END) AS has_disorder_count
    FROM sleep
    GROUP BY occupation, bmi_category
    ORDER BY avg_stress DESC
""", "Occupation Health & Sleep Summary")


📊 Occupation Health & Sleep Summary
          occupation  avg_age  avg_stress  avg_physical_activity  avg_sleep_quality  bmi_category  total_people  has_disorder_count
Sales Representative     28.0        8.00                  30.00               4.00         Obese             2                   2
   Software Engineer     28.0        8.00                  30.00               4.00         Obese             1                   1
          Accountant     52.0        7.00                  45.00               7.00    Overweight             6                   6
Sales Representative     43.5        7.00                  45.00               6.00    Overweight            32                  30
           Scientist     33.5        7.00                  41.00               5.00    Overweight             4                   2
             Teacher     29.0        7.00                  40.00               6.00         Obese             1                   1
              Doctor     31.7        6.

In [9]:
q7=run_query(""" select employee_id,age,gender,job_role, stress_level, sleep_hours, work_hours_per_week
                 from burnout where stress_level>=7 and sleep_hours<=5
                 order by stress_level desc, sleep_hours asc
                 ""","employees with high stress and poor sleep")
                 
                 


📊 employees with high stress and poor sleep
 employee_id  age            gender          job_role  stress_level  sleep_hours  work_hours_per_week
        3958   49            Female Software Engineer          9.99          4.4                   50
        3447   42 Prefer not to say   Project Manager          9.98          4.3                   47
        3558   45              Male   Project Manager          9.98          4.6                   52
        3422   44              Male          IT Admin          9.96          4.2                   43
        2905   23 Prefer not to say Marketing Manager          9.93          4.4                   56
        3618   55            Female Marketing Manager          9.93          5.0                   33
        3979   43 Prefer not to say Marketing Manager          9.85          4.7                   49
        1474   40            Female   Sales Associate          9.84          4.5                   43
        1557   53 Prefer not to say  

In [10]:
q8=run_query(""" select count(*) as total_emp_count
from burnout 
where stress_level>=7 and sleep_hours<=5""","count of q7")


📊 count of q7
 total_emp_count
             202


In [11]:
q8=run_query(""" select count(*) as total_emp_count
from burnout 
""","count of q7")


📊 count of q7
 total_emp_count
            3000


In [12]:
q9 = run_query("""
    SELECT
        employee_id,
        age,
        department,
        country,
        ROUND(
            (stress_level * 0.3) + 
            (burnout_level * 0.3) + 
            ((10 - sleep_hours) * 0.2) + 
            ((work_hours_per_week - 40) * 0.02) +
            ((10 - physical_activity_hrs) * 0.1) +
            (CASE WHEN has_mental_health_support = 'No' THEN 1 ELSE 0 END * 0.1)
        , 2) AS risk_score
    FROM burnout
    WHERE burnout_risk = 1
    ORDER BY risk_score DESC
    LIMIT 10
""", " Top 10 Most At Risk Employees")


📊  Top 10 Most At Risk Employees
 employee_id  age department        country  risk_score
        1557   53         IT United Kingdom        7.69
        1397   25      Sales        Germany        7.68
        1219   30         HR      Australia        7.63
        3610   34      Sales      Australia        7.55
        3957   39    Support United Kingdom        7.55
        2386   58         HR         Canada        7.54
        3927   40  Marketing        Germany        7.54
        3398   41  Marketing        Germany        7.52
        3796   27         IT      Australia        7.52
        1867   35    Support         Brazil        7.49


In [ ]:
from openpyxl import Workbook
import pandas as pd

In [15]:
writer=pd.ExcelWriter('Work_Health_Analysis.xlsx',engine='openpyxl')
#raw data sheets
final_merged_df.to_excel(writer,sheet_name='Burnout_data',index=False)
sleep_df.to_excel(writer,sheet_name='Sleep_data',index=False)
oecd_wide.to_excel(writer,sheet_name='OECD_data',index=False)
#summary sheets using sql results
q5.to_excel(writer,sheet_name='Health and Burnout Summary',index=False)
q6.to_excel(writer,sheet_name='Health and Sleep Summary',index=False)
#save
writer.close()
print("Excel saved!")

Excel saved!
